In [1]:
!pip install pyspark

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = SparkSession.builder.getOrCreate()

In [17]:
video = spark.read.csv('videos-stats.csv', header = True, inferSchema= True )

In [18]:
video.show()
video.printSchema()

+---+--------------------+-----------+------------+-------+--------+--------+-----------+
|_c0|               Title|   Video ID|Published At|Keyword|   Likes|Comments|      Views|
+---+--------------------+-----------+------------+-------+--------+--------+-----------+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech|  3407.0|   672.0|   135612.0|
|  1|The most EXPENSIV...|b3x28s61q3c|  2022-08-24|   tech| 76779.0|  4306.0|  1758063.0|
|  2|My New House Gami...|4mgePWWCAmA|  2022-08-23|   tech| 63825.0|  3338.0|  1564007.0|
|  3|Petrol Vs Liquid ...|kXiYSI7H2b0|  2022-08-23|   tech| 71566.0|  1426.0|   922918.0|
|  4|Best Back to Scho...|ErMwWXQxHp0|  2022-08-08|   tech| 96513.0|  5155.0|  1855644.0|
|  5|Brewmaster Answer...|18fwz9Itbvo|  2021-11-05|   tech| 33570.0|  1643.0|   943119.0|
|  6|Tech Monopolies: ...|jXf04bhcjbg|  2022-06-13|   tech|135047.0|  9367.0|  5937790.0|
|  7|I bought the STRA...|2TqOmtTAMRY|  2022-08-07|   tech|216935.0| 12605.0|  4782514.0|
|  8|15 Em

In [19]:
video_stats_tratar_nulos = video.na.fill({'Likes': 0, 'Comments': 0, 'Views': 0})

video_stats_tratar_nulos.filter(col('Likes') == 0).count()
video_stats_tratar_nulos.filter(col('Comments') == 0).count()
video_stats_tratar_nulos.filter(col('Views') == 0).count()

2

In [20]:
comentario = spark.read.csv('comments.csv', header = True, inferSchema= True )

In [22]:
comentario.show()
comentario.printSchema()

+--------------+-----------+--------------------+------+---------+
|           _c0|   Video ID|             Comment| Likes|Sentiment|
+--------------+-----------+--------------------+------+---------+
|             0|wAZZ-UWGVHI|Let's not forget ...|  95.0|      1.0|
|             1|wAZZ-UWGVHI|Here in NZ 50% of...|  19.0|      0.0|
|             2|wAZZ-UWGVHI|I will forever ac...| 161.0|      2.0|
|             3|wAZZ-UWGVHI|Whenever I go to ...|   8.0|      0.0|
|             4|wAZZ-UWGVHI|Apple Pay is so c...|  34.0|      2.0|
|             5|wAZZ-UWGVHI|We’ve been houndi...|   8.0|      1.0|
|             6|wAZZ-UWGVHI|We only got Apple...|  29.0|      2.0|
|             7|wAZZ-UWGVHI|For now, I need b...|   7.0|      1.0|
|             8|wAZZ-UWGVHI|In the United Sta...|   2.0|      2.0|
|             9|wAZZ-UWGVHI|In Cambodia, we h...|  28.0|      1.0|
|            10|b3x28s61q3c|Wow, you really w...|1344.0|      2.0|
|            11|b3x28s61q3c|The lab is the mo...| 198.0|      

In [23]:
print("Total de Vídeos: ", video.count())
print("Total de Comentários: ", comentario.count())

Total de Vídeos:  1881
Total de Comentários:  30036


In [25]:
video_remove_nulos = video.na.drop(subset = ['Video ID'])
comentario_remove_nulos = comentario.na.drop(subset=['Video ID'])

print('Video ID Removido: - Vídeos ', video_remove_nulos.count())
print('Video ID Removido - Comentarios: ', comentario_remove_nulos.count())

Video ID Removido: - Vídeos  1881
Video ID Removido - Comentarios:  22555


In [27]:
video_remove_duplicada = video.dropDuplicates(['Video ID'])
print("Vídeos Únicos: ", video_remove_duplicada.count())

Vídeos Únicos:  1869


In [30]:
video_tratado = video.\
    withColumn('Likes', col('Likes').cast('int')).\
    withColumn('Comments', col('Comments').cast('int')).\
    withColumn('Views', col('Views').cast('int'))

video_tratado.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)



In [33]:
comentario_tratado = comentario.\
    withColumn('Likes', col('Likes').cast('int')).\
    withColumn('Sentiment', col('Sentiment').cast('int')).\
    withColumnRenamed('Likes', 'Likes Comment')

comentario_tratado.printSchema()
comentario_tratado.show(5)

root
 |-- _c0: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes Comment: integer (nullable = true)
 |-- Sentiment: integer (nullable = true)

+---+-----------+--------------------+-------------+---------+
|_c0|   Video ID|             Comment|Likes Comment|Sentiment|
+---+-----------+--------------------+-------------+---------+
|  0|wAZZ-UWGVHI|Let's not forget ...|           95|        1|
|  1|wAZZ-UWGVHI|Here in NZ 50% of...|           19|        0|
|  2|wAZZ-UWGVHI|I will forever ac...|          161|        2|
|  3|wAZZ-UWGVHI|Whenever I go to ...|            8|        0|
|  4|wAZZ-UWGVHI|Apple Pay is so c...|           34|        2|
+---+-----------+--------------------+-------------+---------+
only showing top 5 rows



In [34]:
video_tratado = video.\
    withColumn('Interaction', col('Likes') + col('Comments') + col('Views'))

video_tratado.show(5)

+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+
|_c0|               Title|   Video ID|Published At|Keyword|  Likes|Comments|    Views|Interaction|
+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|
|  1|The most EXPENSIV...|b3x28s61q3c|  2022-08-24|   tech|76779.0|  4306.0|1758063.0|  1839148.0|
|  2|My New House Gami...|4mgePWWCAmA|  2022-08-23|   tech|63825.0|  3338.0|1564007.0|  1631170.0|
|  3|Petrol Vs Liquid ...|kXiYSI7H2b0|  2022-08-23|   tech|71566.0|  1426.0| 922918.0|   995910.0|
|  4|Best Back to Scho...|ErMwWXQxHp0|  2022-08-08|   tech|96513.0|  5155.0|1855644.0|  1957312.0|
+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+
only showing top 5 rows



In [37]:
video_data_df = video_tratado.withColumn('Year', year(col('Published At')))
video_data_df.show(5)

+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
|_c0|               Title|   Video ID|Published At|Keyword|  Likes|Comments|    Views|Interaction|Year|
+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|
|  1|The most EXPENSIV...|b3x28s61q3c|  2022-08-24|   tech|76779.0|  4306.0|1758063.0|  1839148.0|2022|
|  2|My New House Gami...|4mgePWWCAmA|  2022-08-23|   tech|63825.0|  3338.0|1564007.0|  1631170.0|2022|
|  3|Petrol Vs Liquid ...|kXiYSI7H2b0|  2022-08-23|   tech|71566.0|  1426.0| 922918.0|   995910.0|2022|
|  4|Best Back to Scho...|ErMwWXQxHp0|  2022-08-08|   tech|96513.0|  5155.0|1855644.0|  1957312.0|2022|
+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
only showing top 5 rows



In [42]:
df_join_video_comments = video_data_df.join(comentario_tratado, 'Video ID')

video_data_df.show(5)
comentario_tratado.show(5)
df_join_video_comments.show(5)

+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
|_c0|               Title|   Video ID|Published At|Keyword|  Likes|Comments|    Views|Interaction|Year|
+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|
|  1|The most EXPENSIV...|b3x28s61q3c|  2022-08-24|   tech|76779.0|  4306.0|1758063.0|  1839148.0|2022|
|  2|My New House Gami...|4mgePWWCAmA|  2022-08-23|   tech|63825.0|  3338.0|1564007.0|  1631170.0|2022|
|  3|Petrol Vs Liquid ...|kXiYSI7H2b0|  2022-08-23|   tech|71566.0|  1426.0| 922918.0|   995910.0|2022|
|  4|Best Back to Scho...|ErMwWXQxHp0|  2022-08-08|   tech|96513.0|  5155.0|1855644.0|  1957312.0|2022|
+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
only showing top 5 rows

+---+-----------+--------------------+-

In [45]:
us_videos = spark.read.csv('USvideos.csv', header = True, inferSchema= True )

us_videos.show(5)
us_videos.printSchema()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

In [51]:
us_videos = us_videos.withColumnRenamed('title', 'Title')

df_join_video_usvideos = video_data_df.join(us_videos, 'Title')

video_data_df.show(5)
us_videos.show(5)
df_join_video_usvideos.show(5)

+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
|_c0|               Title|   Video ID|Published At|Keyword|  Likes|Comments|    Views|Interaction|Year|
+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|
|  1|The most EXPENSIV...|b3x28s61q3c|  2022-08-24|   tech|76779.0|  4306.0|1758063.0|  1839148.0|2022|
|  2|My New House Gami...|4mgePWWCAmA|  2022-08-23|   tech|63825.0|  3338.0|1564007.0|  1631170.0|2022|
|  3|Petrol Vs Liquid ...|kXiYSI7H2b0|  2022-08-23|   tech|71566.0|  1426.0| 922918.0|   995910.0|2022|
|  4|Best Back to Scho...|ErMwWXQxHp0|  2022-08-08|   tech|96513.0|  5155.0|1855644.0|  1957312.0|2022|
+---+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
only showing top 5 rows

+-----------+-------------+------------

In [52]:
for coluna in video.columns:
    video.select(count(when(col(coluna).isNull(), coluna)).alias(coluna)).show()

+---+
|_c0|
+---+
|  0|
+---+

+-----+
|Title|
+-----+
|    0|
+-----+

+--------+
|Video ID|
+--------+
|       0|
+--------+

+------------+
|Published At|
+------------+
|           0|
+------------+

+-------+
|Keyword|
+-------+
|      0|
+-------+

+-----+
|Likes|
+-----+
|    2|
+-----+

+--------+
|Comments|
+--------+
|       2|
+--------+

+-----+
|Views|
+-----+
|    2|
+-----+



In [54]:
video_tratado_2 = video_data_df.drop('_c0')

video_tratado_2.show(5)

+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
|               Title|   Video ID|Published At|Keyword|  Likes|Comments|    Views|Interaction|Year|
+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|
|The most EXPENSIV...|b3x28s61q3c|  2022-08-24|   tech|76779.0|  4306.0|1758063.0|  1839148.0|2022|
|My New House Gami...|4mgePWWCAmA|  2022-08-23|   tech|63825.0|  3338.0|1564007.0|  1631170.0|2022|
|Petrol Vs Liquid ...|kXiYSI7H2b0|  2022-08-23|   tech|71566.0|  1426.0| 922918.0|   995910.0|2022|
|Best Back to Scho...|ErMwWXQxHp0|  2022-08-08|   tech|96513.0|  5155.0|1855644.0|  1957312.0|2022|
+--------------------+-----------+------------+-------+-------+--------+---------+-----------+----+
only showing top 5 rows



In [55]:
video_tratado_2.write.mode('overwrite').option('header', 'true').parquet('video-tratados-parquet.parquet')

In [56]:
df_join_video_comments = df_join_video_comments.drop('_c0')

df_join_video_comments.show(5)

+-----------+--------------------+------------+-------+------+--------+--------+-----------+----+--------------------+-------------+---------+
|   Video ID|               Title|Published At|Keyword| Likes|Comments|   Views|Interaction|Year|             Comment|Likes Comment|Sentiment|
+-----------+--------------------+------------+-------+------+--------+--------+-----------+----+--------------------+-------------+---------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech|3407.0|   672.0|135612.0|   139691.0|2022|Let's not forget ...|           95|        1|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech|3407.0|   672.0|135612.0|   139691.0|2022|Here in NZ 50% of...|           19|        0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech|3407.0|   672.0|135612.0|   139691.0|2022|I will forever ac...|          161|        2|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech|3407.0|   672.0|135612.0|   139691.0|2022|Whenever I go to ...|            8|        0|

In [57]:
df_join_video_comments.write.mode('overwrite').option('header', 'true').parquet('videos-comments-tratados.parquet')

In [58]:
spark.stop()